## 1. Data Representation, Preprocessing, and Standard Implementation
**Data Preprocessing & Representation**
To prepare the raw abstract texts for the Naive Bayes algorithm, several preprocessing steps were applied to ensure clean and meaningful features.
First, the target column `class` was renamed to `target_domain` to avoid naming conflicts with the word "class" potentially occurring in the text. Second, to address the strict conditional independence assumption of Naive Bayes, specific biological binomial nomenclatures mentioned in the assignment prompt (e.g., *"homo sapiens"*, *"escherichia coli"*, *"human immunodeficiency virus"*) were concatenated using hyphens (e.g., *"homo-sapiens"*). This prevents the model from treating these highly correlated word pairs as independent pieces of evidence. All text was lowercased.
For the standard model, the text was converted into a Bag-of-Words (BoW) representation using `CountVectorizer` with `binary=True` and `stop_words='english'`, creating a sparse matrix of 0-1 attributes indicating the presence or absence of words while filtering out uninformative English stop words.

**Standard Naive Bayes Implementation**
The standard Naive Bayes classifier was implemented completely from scratch using `numpy`. To avoid floating-point underflow issues caused by multiplying many small probabilities, the calculations were performed in log-space (adding logarithms instead of multiplying raw probabilities). Additionally, Laplace smoothing (add-one smoothing) was applied during the calculation of conditional probabilities $P(w|C)$ to handle out-of-vocabulary words and prevent zero-probability outputs for previously unseen features.

## 2. Model Extensions and Modifications
To explore enhancements to the standard model and tackle the poor assumptions of Naive Bayes text classifiers, the following modifications were introduced into the data representation phase:
1. **Introduction of N-grams:** The strict conditional independence assumption ignores word order. By extracting both unigrams and bi-grams (`ngram_range=(1, 2)`), the extended model attempts to capture local contexts and meaningful biological phrases (e.g., "cell membrane") that are split in standard BoW.
2. **Noise Reduction via Minimum Document Frequency:** A `min_df` threshold was introduced. The rationale was that words appearing in very few documents are often typos or random noise, and filtering them out could reduce the dimensionality of the feature space and prevent overfitting.

## 3. Hyper-parameters Tuning
The hyperparameters primarily govern the feature extraction process. **Their optimal values were empirically determined by evaluating various combinations (manual grid search) on the 20% validation set to find the best trade-off between capturing biological context and avoiding feature sparsity.** They were tuned as follows:
* **`binary`**: Kept at `True` for both models to strictly follow the 0-1 attribute requirement requested in Task 1.
* **`stop_words`**: Set to `'english'` to automatically remove uninformative words, improving computational efficiency.
* **`ngram_range`**: Set to `(1, 1)` for the Standard Model. For the Extended Model, we tested `(1, 2)` and `(1, 3)`, and found that `(1, 2)` (unigrams and bi-grams) yielded the most stable validation performance without causing extreme memory overhead.
* **`min_df`**: Set to `1` (default) for the Standard Model. For the Extended Model, we experimented with thresholds of 2, 3, and 5. Setting it to `3` was chosen as it effectively filtered out misspelled words and ultra-rare noise while preserving important domain-specific terminology.

## 4. Evaluation Procedure
To evaluate the models without overfitting the blind test set, the provided training data was split into a **training set (80%)** and a **validation set (20%)** using `train_test_split` with a fixed random state for reproducibility.
The standard model and the extended model were both trained on the 80% split and evaluated on the 20% split. A Baseline "Null Model" was also created by predicting the majority class of the training set for all validation instances. Finally, to generate the ultimate Kaggle submission, the best-performing model architecture was retrained on **100%** of the available training data to maximize its learning capacity before predicting the blind `tst.csv` labels.

## 5. Results and Analysis
The evaluation yielded the following accuracy scores on the 20% validation split:
* **Null Model (Baseline) Accuracy:** 50.62%
* **Standard Naive Bayes Accuracy:** 92.75%
* **Extended Naive Bayes (N-grams + min_df):** 86.88%
* **Kaggle Public Leaderboard Score (Final Standard Model):** 0.940 (94.00%)

**Analysis:**
The results demonstrate the extreme effectiveness of the standard Naive Bayes algorithm for this specific text classification task. The Standard NB model (92.75%) vastly outperformed the Baseline Null Model (50.62%), proving that the presence of specific independent terminology is a highly reliable predictor of the biological domain.

Interestingly, the Extended Model saw a drop in accuracy to 86.88%. This highlights a classic machine learning trade-off. By introducing bi-grams, we exponentially increased the feature space, leading to a highly sparse matrix (the Curse of Dimensionality) which likely caused the model to overfit the training data. Furthermore, applying `min_df=3` may have aggressively stripped away rare but highly discriminative biological terms that the standard model successfully utilized.

Consequently, the simpler **Standard Naive Bayes** proved to be the superior and more robust architecture for this specific dataset size. The final Kaggle public score of **0.940** strongly confirms our hypothesis, demonstrating that the standard model generalizes exceptionally well to the unseen blind test set.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ==========================================
# Task 1: Data Preprocessing & Representation
# ==========================================

# 1. Load the data
train_df = pd.read_csv('trg.csv')
test_df = pd.read_csv('tst.csv')

# 2. [Crucial Step] Rename the target column from 'class' to 'target_domain'
# to avoid conflicts with the word 'class' in the abstracts.
train_df.rename(columns={'class': 'target_domain'}, inplace=True)

# 3. Text preprocessing function (handles terms violating the independence assumption)
def clean_text(text):
    text = text.lower()
    # Concatenate biological phrases using hyphens so they are treated as a single feature
    text = text.replace('homo sapiens', 'homo-sapiens')
    text = text.replace('escherichia coli', 'escherichia-coli')
    text = text.replace('human immunodeficiency virus', 'human-immunodeficiency-virus')
    return text

train_df['abstract_cleaned'] = train_df['abstract'].apply(clean_text)
test_df['abstract_cleaned'] = test_df['abstract'].apply(clean_text)

# 4. Split into training and validation sets for internal evaluation (80/20 split)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    train_df['abstract_cleaned'],
    train_df['target_domain'],
    test_size=0.2,
    random_state=42
)

# 5. Feature Extraction: Standard Bag of Words (0-1 attributes)
vectorizer = CountVectorizer(binary=True, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train_raw).toarray()
X_val_vec = vectorizer.transform(X_val_raw).toarray() # Only transform the validation set!


# ==========================================
# Task 2: Standard Naive Bayes Implementation
# ==========================================

class MyNaiveBayes:
    def __init__(self):
        self.classes = None
        self.log_priors = {}
        self.log_likelihoods = {}

    def fit(self, X, y):
        self.classes = np.unique(y)
        total_docs = X.shape[0]
        vocab_size = X.shape[1]

        for c in self.classes:
            # Extract all samples belonging to class c
            X_c = X[y == c]

            # 1. Calculate prior probabilities P(C) in log space: log( N_c / N_total )
            self.log_priors[c] = np.log(X_c.shape[0] / total_docs)

            # 2. Calculate conditional probabilities P(w|C) using Laplace Smoothing
            # Total occurrences of each word in class c
            word_counts_c = X_c.sum(axis=0)
            total_words_c = word_counts_c.sum()

            # Apply add-one (+1) smoothing and calculate log likelihoods
            # Formula: log[ (count(w, c) + 1) / (total_words_c + |V|) ]
            likelihood_c = (word_counts_c + 1) / (total_words_c + vocab_size)
            self.log_likelihoods[c] = np.log(likelihood_c)

    def predict(self, X):
        predictions = []
        for doc in X:
            best_class = None
            max_log_prob = -np.inf

            for c in self.classes:
                # Naive Bayes formula in log space: log P(C) + sum( log P(w|C) * x_i )
                # 'doc' is an array of 0s and 1s, so the dot product effectively sums
                # the log probabilities of only the words present in the document.
                log_prob = self.log_priors[c] + np.sum(doc * self.log_likelihoods[c])

                if log_prob > max_log_prob:
                    max_log_prob = log_prob
                    best_class = c

            predictions.append(best_class)
        return np.array(predictions)


# ==========================================
# Task 3: Evaluation & Baseline (Null Model)
# ==========================================

# 1. Train the standard implementation
nb_model = MyNaiveBayes()
nb_model.fit(X_train_vec, y_train.values)

# 2. Predict and evaluate on the validation set
val_preds = nb_model.predict(X_val_vec)
nb_accuracy = accuracy_score(y_val, val_preds)

# 3. Calculate the Null Model (Baseline) accuracy
# The Null Model always predicts the most frequent class in the training set
majority_class = y_train.value_counts().idxmax()
null_preds = [majority_class] * len(y_val)
null_accuracy = accuracy_score(y_val, null_preds)

print(f"Null Model (Baseline) Accuracy: {null_accuracy * 100:.2f}%")
print(f"Standard Naive Bayes Accuracy: {nb_accuracy * 100:.2f}%")


# ==========================================
# Task 4: Improved Model & Kaggle Submission
# ==========================================

# 1. Improved feature extraction (adding bi-grams and filtering noise)
improved_vectorizer = CountVectorizer(
    binary=True,
    stop_words='english',
    ngram_range=(1, 2), # Improvement 1: Include 2-grams (captures local context)
    min_df=3            # Improvement 2: Filter extreme low-frequency noise words
)

# Extract features again for validation comparison
X_train_imp = improved_vectorizer.fit_transform(X_train_raw).toarray()
X_val_imp = improved_vectorizer.transform(X_val_raw).toarray()

# Train and evaluate the improved model
improved_model = MyNaiveBayes()
improved_model.fit(X_train_imp, y_train.values)
val_preds_imp = improved_model.predict(X_val_imp)
imp_accuracy = accuracy_score(y_val, val_preds_imp)

print(f"Improved Naive Bayes Accuracy: {imp_accuracy * 100:.2f}%")

# ---------------------------------------------------------
# 2. Final Training on 100% of the training data for Kaggle
# AS PER REPORT: We use the Standard Model since it performed better (92.75% vs 86.88%)
# ---------------------------------------------------------
X_full_train_std = vectorizer.fit_transform(train_df['abstract_cleaned']).toarray()
X_test_final_std = vectorizer.transform(test_df['abstract_cleaned']).toarray()

final_standard_model = MyNaiveBayes()
final_standard_model.fit(X_full_train_std, train_df['target_domain'].values)

# Predict the blind test set using the winning Standard Model
final_test_predictions = final_standard_model.predict(X_test_final_std)

# 3. Format the submission DataFrame as required by Kaggle (id, class)
final_submission = pd.DataFrame({
    'id': test_df['id'],
    'class': final_test_predictions
})

# Export to CSV without the index column
final_submission.to_csv('best_nb_submission.csv', index=False)
print("Final submission file generated successfully using the Standard Model: best_nb_submission.csv")

Null Model (Baseline) Accuracy: 50.62%
Standard Naive Bayes Accuracy: 92.75%
Improved Naive Bayes Accuracy: 86.88%
Final submission file generated successfully using the Standard Model: best_nb_submission.csv


In [ ]:
!jupyter nbconvert --to html CS361_HW2_Pan.ipynb

[NbConvertApp] Converting notebook CS361_HW2_Pan.ipynb to html
[NbConvertApp] Writing 309075 bytes to CS361_HW2_Pan.html
